# نَبْض | تسجيل الكتاب والملزمة بصوت المعلّمة

هذا الدفتر مجهز لحزمة مشروعك الموجودة **مسبقاً داخل Google Drive الخاص بك**. لا تحتاج ترفع الملف ولا تكتب أي أوامر.

1. من أعلى الشاشة: **Runtime → Run all** / **تشغيل الكل**.
2. وافق على ربط Google Drive إذا ظهر طلب.
3. سيُنشئ تسجيل اختبار ثم يبدأ تسجيل **دفعة تجريبية صغيرة**؛ جميع الناتج مسودات تُراجع قبل النشر.

**مهم:** الصوت مطابق على سبيل المحاولة فقط وليس مضموناً. مصدر الكتاب والملزمة لا يغني عن تدقيق كل حرف. التحميل من الإنترنت مطلوب لتنزيل نموذج الصوت، وقد يستغرق فترة. لا يُستخدم أي رصيد توليد مدفوع من هذا الدفتر. الجلسة قابلة للانقطاع؛ الملفات المكتملة تبقى محفوظة في Drive.


## ١ — تثبيت محرّك الصوت المفتوح المصدر
يُحمّل النموذج الرسمي متعدد اللغات، مع دعم العربية.


In [ ]:
import sys, subprocess, importlib.util, shutil
assert shutil.which('ffmpeg'), 'ffmpeg غير موجود في بيئة كولاب'
if importlib.util.find_spec('chatterbox') is None:
    print('تثبيت محرّك الصوت...')
    subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/resemble-ai/chatterbox.git','soundfile'],check=True)
import torch
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('المعالج المستخدم:',DEVICE, ' — عند عدم توفر GPU يستخدم المعالج العادي وهو أبطأ')
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from google.colab import drive
from IPython.display import Audio, display
import torchaudio as ta


## ٢ — فتح الحزمة الموجودة في Google Drive
الملف محفوظ بمجلد خاص باسم `Nabd Biology Audio - private`، وليس منشوراً على GitHub.


In [ ]:
from pathlib import Path
import zipfile, json, hashlib, sys
DRIVE_DIR=Path('/content/drive/MyDrive/Nabd Biology Audio - private')
drive.mount('/content/drive')
SOURCE_ZIP=DRIVE_DIR/'Biology_Audio_App_v8.zip'
if not SOURCE_ZIP.exists():
    raise FileNotFoundError('لم أجد حزمة الأحياء في Google Drive: '+str(SOURCE_ZIP))
ROOT=DRIVE_DIR/'working_V8'
ROOT.mkdir(parents=True,exist_ok=True)
if not (ROOT/'manifest.json').exists():
    with zipfile.ZipFile(SOURCE_ZIP) as src:
        base=ROOT.resolve()
        for entry in src.infolist():
            target=(ROOT/entry.filename).resolve()
            if target!=base and base not in target.parents: raise ValueError('مسار غير آمن داخل ZIP')
        src.extractall(ROOT)
m=json.loads((ROOT/'manifest.json').read_text('utf-8'))
assert len(m['segments'])==1051, 'حزمة غير مكتملة'
for d in m['documents']:
    h=hashlib.sha256((ROOT/d['pdf_file']).read_bytes()).hexdigest()
    assert h==d['source_pdf_sha256'], 'المصدر الأصلي غير مطابق: '+d['id']
sys.path.insert(0,str(ROOT/'tools'))
from generate_voice_batch import preflight, run
print('تم التحقق من',sum(d['pdf_pages'] for d in m['documents']),'صفحة و',len(m['segments']),'مقطع نصي')
print(json.dumps(preflight(ROOT,allow_unverified=True,max_new=3),ensure_ascii=False,indent=2))


## ٣ — إنتاج اختبار حقيقي بصوت العينة
النتيجة هنا **تجربة مستقلة** وليست قراءة معتمدة. استمع إليها لمعرفة مدى قرب صوتها من صوت المعلّمة.


In [ ]:
import gc, inspect
REF=ROOT/'audio/reference_voice_9s.wav'
assert REF.exists(),'التسجيل المرجعي غير موجود'
print('الصوت المرجعي الأصلي:')
display(Audio(filename=str(REF)))
if 't3_model' not in inspect.signature(ChatterboxMultilingualTTS.from_pretrained).parameters:
    raise RuntimeError('نسخة قديمة من النموذج، أعد تشغيل الجلسة ثم تشغيل الكل')
MODEL=ChatterboxMultilingualTTS.from_pretrained(device=DEVICE,t3_model='v3')
TEST='هسه نجي للجهاز العصبي. الخلية العصبية هي الوحدة الأساسية في بنائه.'
with torch.inference_mode():
    w=MODEL.generate(TEST,language_id='ar',audio_prompt_path=str(REF),cfg_weight=0.3,exaggeration=0.5)
trial=ROOT/'audio/new_voice_trial.wav'
ta.save(str(trial),w.detach().cpu().unsqueeze(0) if w.ndim==1 else w.detach().cpu(),MODEL.sr)
print('الصوت الجديد التجريبي (ليس معتمداً):')
display(Audio(filename=str(trial)))
del MODEL,w
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('حُفظ التسجيل التجريبي في Google Drive:',trial)


## ٤ — إنتاج أول 10 مقاطع كمسودات
كل تسجيل يُحفظ فور توليده. إعادة التشغيل تستكمل من المقاطع الناجحة. لا نعرضها للطلاب على أنها معتمدة إلا بعد التدقيق النصي والاستماع.


In [ ]:
from generate_voice_batch import run
try:
    count=run(ROOT,REF,max_new=10,source='textbook',start_page=1,
              allow_unverified=True,include_flagged=False,device=DEVICE)
    print('مقاطع جديدة محفوظة:',count)
except Exception as e:
    print('حدث مانع في التوليد، ولن تضيع المقاطع التي حفظت:',type(e).__name__,str(e))
    raise


## ٥ — عرض النتيجة وحالة التدقيق
التوليد لا يساوي الاعتماد النهائي. لم يتم تدقيق النص حرفياً ولا سماع المقاطع كلّها بعد.


In [ ]:
from collections import Counter
m=json.loads((ROOT/'manifest.json').read_text('utf-8'))
print('حالات الصوت:',dict(Counter(s['audio_status'] for s in m['segments'])))
print('مسودات MP3 المحفوظة:',len(list((ROOT/'audio/generated').glob('*.mp3'))))
print('المسار:',ROOT/'audio/generated')
print('صفحات التدقيق:')
print(ROOT/'review_text.html')
print(ROOT/'review_audio.html')
